# Building a RAG and Architecture Patterns

This notebook moves from embeddings as **model inputs** to embeddings as **application infrastructure**.

In the previous LLM material, embeddings appeared inside the model: token IDs were mapped into vectors so a Transformer could process text. In this notebook, we use a similar idea outside the model. We represent documents and questions as vectors, compare them, and retrieve the pieces of text that are most relevant to a user question.

That idea is the core of **Retrieval-Augmented Generation (RAG)**:

```text
retrieve useful evidence -> place it in the prompt -> generate an answer grounded in that evidence
```

We will build a small local RAG pipeline without external APIs. The implementation is intentionally simple so every step is visible. After that, we will use the same components to introduce common LLM architecture patterns: prompt-only systems, RAG systems, tool-using agents, planner-executor workflows, evaluator loops, and multi-agent designs.


## Learning Objectives

By the end of this notebook you should be able to:

- Explain why prompt-only systems are limited.
- Describe the main components of a RAG system.
- Chunk small documents and attach useful metadata.
- Build simple local embeddings for retrieval.
- Retrieve relevant chunks with cosine similarity.
- Compose a grounded prompt from retrieved context.
- Explain why retrieval quality affects answer quality.
- Compare prompt-only, RAG, tool-using, planner-executor, reflection, and multi-agent patterns.

A good way to read this notebook is to keep asking three questions:

1. What information does the system have?
2. How does the system decide which information is relevant?
3. What responsibility belongs to retrieval, and what responsibility belongs to generation?


## 0. Setup

This notebook intentionally avoids external LLM APIs. That makes the pipeline easier to understand and easier to run in class.

In a production RAG system, we would usually use:

- A strong embedding model instead of the toy bag-of-words embedding used here.
- A vector database instead of a Python list.
- A real LLM call instead of the rule-based answerer later in the notebook.
- A tracing and evaluation system to inspect failures.

The architecture, however, is the same. We are learning the skeleton before replacing the simple parts with stronger production components.


In [ ]:
import math
import re
from collections import Counter, defaultdict

try:
    import numpy as np
except ImportError:
    np = None

print("Setup complete")
print("numpy available:", np is not None)


## 1. Why Prompting Alone Is Not Enough

A prompt-only application sends the user's question directly to the model. This can work well when the task is self-contained or when the model already has enough knowledge in its parameters.

For example, a prompt-only model can usually help rewrite a paragraph, summarize short text pasted into the prompt, or explain a general concept. But it breaks down when the answer depends on information the model cannot reliably know at generation time.

Prompt-only systems are limited when the answer depends on:

- **Private documents:** course notes, company policies, internal manuals, contracts, or student submissions.
- **Recent or changing information:** schedules, prices, APIs, current policies, or live data.
- **Long source material:** content that does not fit comfortably into the prompt.
- **Exact citations:** answers that must point back to specific evidence.
- **Domain-specific procedures:** steps that are not part of general internet-scale knowledge.

Retrieval-Augmented Generation, or **RAG**, adds a retrieval step before generation. The model is not asked to answer from memory alone. It receives relevant evidence as part of the prompt.


### Concept Note: Prompt-Only vs RAG

Prompt-only:

```text
user question -> LLM -> answer
```

RAG:

```text
user question -> retriever -> relevant context -> LLM -> grounded answer
```

The important shift is this: in a RAG system, the LLM is not the only source of knowledge. The system also has a document collection, a retriever, and a prompt-composition step.

This gives us a useful separation of responsibilities:

| Component | Responsibility |
| --- | --- |
| Retriever | Find useful evidence. |
| Prompt builder | Place evidence and instructions into the model input. |
| Generator | Write the final answer using the provided context. |
| Evaluator | Check whether the answer is grounded and useful. |


## 2. A Tiny Document Collection

A real RAG system usually starts with PDFs, web pages, database records, notebooks, manuals, or other long-form documents. To keep the mechanics visible, we will use five short course snippets.

Each document has three fields:

- `id`: a stable identifier used by the system.
- `title`: a human-readable name.
- `text`: the content we want to retrieve from.

The examples are intentionally small. This lets us inspect the full document collection, the chunks, the vocabulary, and the retrieval scores without hiding anything behind a library.


In [ ]:
documents = [
    {
        "id": "nn-foundations",
        "title": "Neural Networks Foundations",
        "text": "A neural network is built from layers of neurons. Each layer transforms input features into representations that can support prediction. Training adjusts weights with gradient descent."
    },
    {
        "id": "embeddings",
        "title": "Embeddings",
        "text": "Embeddings map discrete tokens or items into dense vectors. In language models, token embeddings are learned lookup tables indexed by token IDs."
    },
    {
        "id": "positional-embeddings",
        "title": "Positional Embeddings",
        "text": "A language model needs order information because the same tokens can mean different things in different positions. Positional embeddings add sequence position information to token representations."
    },
    {
        "id": "rag",
        "title": "Retrieval-Augmented Generation",
        "text": "RAG systems retrieve relevant document chunks before generating an answer. The retrieved context helps the model answer with information outside its parameters."
    },
    {
        "id": "agents",
        "title": "Agentic Patterns",
        "text": "Agentic systems combine language models with tools, memory, planning, and evaluation loops. The architecture should stay as simple as the task allows."
    },
]

for doc in documents:
    print(f"{doc['id']}: {doc['title']}")


### Reading the Document Objects

The code above creates a list of dictionaries. This is a miniature document store.

In a real application, these records might come from uploaded PDFs, Markdown files, database rows, HTML pages, or notebook exports. The important idea is that each source should have stable metadata before indexing begins.

If metadata is weak at this stage, citations and debugging become difficult later.


## 3. Chunking

A **chunk** is the unit we retrieve.

Most source documents are too long to retrieve as a whole. If we retrieve an entire manual or notebook for every question, the context becomes noisy and expensive. Instead, we split documents into smaller pieces and retrieve only the pieces that seem relevant.

Chunking is one of the most important RAG design choices:

- If chunks are too small, they may lose the surrounding explanation needed to answer a question.
- If chunks are too large, they may include too many unrelated ideas.
- If chunks do not overlap, an important idea can be split across two chunks.
- If chunks overlap too much, the retriever may return redundant context.

In this notebook, `chunk_document` splits text by word count and keeps a small overlap between consecutive chunks. Production systems often use more advanced strategies: paragraph boundaries, Markdown headers, semantic sectioning, page numbers, or notebook cell metadata.

The `chunk_id` matters because it becomes the citation handle later. When the answer cites `[rag:0]`, it is pointing back to a specific retrieved chunk.


In [ ]:
STOPWORDS = {
    "a", "an", "and", "are", "as", "by", "can", "do", "does", "from", "how",
    "in", "is", "it", "of", "or", "the", "to", "what", "when", "where", "why",
    "with"
}

def tokenize(text):
    return [
        token for token in re.findall(r"[a-z]+", text.lower())
        if token not in STOPWORDS
    ]

def chunk_document(doc, max_words=24, overlap=6):
    words = doc["text"].split()
    chunks = []
    step = max_words - overlap
    for start in range(0, len(words), step):
        piece = words[start:start + max_words]
        if not piece:
            continue
        chunks.append({
            "doc_id": doc["id"],
            "title": doc["title"],
            "chunk_id": f"{doc['id']}:{len(chunks)}",
            "text": " ".join(piece)
        })
        if start + max_words >= len(words):
            break
    return chunks

chunks = []
for doc in documents:
    chunks.extend(chunk_document(doc))

print("Number of chunks:", len(chunks))
for chunk in chunks:
    print(f"[{chunk['chunk_id']}] {chunk['text']}")


### Reading the Chunking Output

After running the chunking cell, inspect the printed chunks.

Notice that each chunk keeps:

- The original document ID.
- The title.
- A chunk ID.
- The chunk text.

This metadata is not decoration. It is what allows the final answer to cite evidence and what allows developers to debug retrieval failures.


### Mini Lab 1

Change `max_words` and `overlap` in the chunking cell.

Observe:

- How many chunks are produced?
- Do the chunks still contain enough information to answer a question?
- Do consecutive chunks repeat too much information?
- Are any important ideas split in an awkward way?

Discussion questions:

- When do chunks become too short to answer a question?
- When do chunks start mixing too many ideas?
- What metadata would be useful for real course PDFs, slides, or notebooks?
- How would you preserve source information such as filename, page number, slide number, or notebook section?


## 4. A Transparent Local Embedding

A retriever needs a way to compare a user question with document chunks. To do that, we represent both as vectors.

For teaching purposes, we will use a **bag-of-words embedding**:

1. Build a vocabulary from the words that appear in the chunks.
2. Represent each chunk as a vector of word counts.
3. Normalize the vector so longer chunks do not win only because they have more words.
4. Represent the question with the same vocabulary.
5. Compare the question vector with each chunk vector.

This is not a state-of-the-art embedding. It does not understand synonyms very well. For example, it may not know that "retrieve" and "search" are related unless both words appear in the text. That limitation is useful pedagogically: students can see why production systems use learned embedding models.

Still, the basic retrieval idea is the same:

```text
represent question as vector
represent chunks as vectors
return chunks whose vectors are closest to the question vector
```


In [ ]:
vocabulary = sorted({tok for chunk in chunks for tok in tokenize(chunk["text"])})
tok_to_idx = {tok: i for i, tok in enumerate(vocabulary)}

print("Vocabulary size:", len(vocabulary))
print(vocabulary[:30])


### Reading the Vocabulary

The vocabulary is the set of terms the toy retriever can recognize. If a word does not appear in the vocabulary, it cannot contribute to similarity.

This is one reason bag-of-words retrieval is limited. A learned embedding model can often connect related words even when the exact terms differ. For example, a stronger model may understand that "lookup table" and "embedding matrix" are related.


In [ ]:
def bow_embedding(text):
    counts = Counter(tokenize(text))
    if np is not None:
        vec = np.zeros(len(vocabulary), dtype=float)
        for tok, count in counts.items():
            if tok in tok_to_idx:
                vec[tok_to_idx[tok]] = count
        norm = np.linalg.norm(vec)
        return vec if norm == 0 else vec / norm
    else:
        vec = {tok_to_idx[tok]: count for tok, count in counts.items() if tok in tok_to_idx}
        norm = math.sqrt(sum(v * v for v in vec.values()))
        return {i: v / norm for i, v in vec.items()} if norm else vec

def cosine(a, b):
    if np is not None:
        return float(np.dot(a, b))
    return sum(a.get(i, 0.0) * b.get(i, 0.0) for i in set(a) | set(b))

chunk_embeddings = [bow_embedding(chunk["text"]) for chunk in chunks]
print("Built", len(chunk_embeddings), "chunk embeddings")


### Reading the Embedding Code

The `bow_embedding` function has two implementations:

- If `numpy` is installed, it uses dense vectors.
- If `numpy` is not installed, it uses sparse dictionaries.

Both versions represent the same idea: each text becomes a vector indexed by vocabulary terms.

The `cosine` function compares two vectors. If two texts share important vocabulary, their cosine similarity is higher. If they share few or no terms, their similarity is lower.


## 5. Retrieval

Retrieval compares the question vector with every chunk vector and returns the most similar chunks.

The similarity measure here is **cosine similarity**. Intuitively, cosine similarity asks whether two vectors point in a similar direction. In text retrieval, that usually means they share important terms or concepts.

The retriever returns a ranked list:

```text
score, chunk
score, chunk
score, chunk
```

Higher scores mean the chunk is more similar to the question according to our embedding method. The scores are not perfect truth; they are signals. A real RAG system should always be evaluated because the retriever can return irrelevant or incomplete evidence.


In [ ]:
def retrieve(question, top_k=3):
    q = bow_embedding(question)
    scored = []
    for chunk, emb in zip(chunks, chunk_embeddings):
        scored.append((cosine(q, emb), chunk))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]

question = "Why do language models need positional embeddings?"
for score, chunk in retrieve(question):
    print(f"score={score:.3f} [{chunk['chunk_id']}] {chunk['text']}")


### Reading the Retrieval Output

Each printed line contains three things:

```text
similarity score, chunk ID, chunk text
```

The chunk ID is the evidence handle. The score is the retriever's estimate of relevance. The text is the actual evidence that would be sent to the generator.

Do not treat the score as an absolute truth. A score is only meaningful relative to the embedding method and the other chunks in the index.


### Checkpoint

Try these questions:

- What are embeddings?
- How does RAG use documents?
- Why are tools useful for agents?
- How are neural networks trained?

For each question, inspect the top retrieved chunks.

Ask:

- Did the retriever find the chunk a human would expect?
- Did it retrieve a chunk with direct evidence, or only a vaguely related one?
- Did it miss a relevant chunk because the wording was different?
- What would happen if we changed the question wording?

This is the core habit for debugging RAG: do not start by blaming the LLM. First inspect what evidence the retriever actually provided.


## 6. Prompt Composition

A RAG system usually inserts retrieved context into the prompt before asking the model to answer.

This step is more important than it first appears. The prompt must tell the model:

- What role it should play.
- What context it is allowed to use.
- What to do if the context is insufficient.
- How to cite evidence.
- What format the final answer should follow.

In this notebook, we build the prompt but do not call an external LLM. The point is to see the structure clearly.

A useful RAG prompt often contains four parts:

1. **Instruction:** what the model should do.
2. **Grounding rule:** use only the retrieved context.
3. **Context:** the retrieved chunks with citation IDs.
4. **Question:** the user's actual question.


In [ ]:
def build_rag_prompt(question, retrieved):
    context_blocks = []
    for score, chunk in retrieved:
        context_blocks.append(
            f"[{chunk['chunk_id']}] {chunk['title']}\n{chunk['text']}"
        )
    context = "\n\n".join(context_blocks)
    return f"""You are a course assistant. Answer the question using only the context below.
If the context is insufficient, say what is missing. Cite chunk IDs in square brackets.

Context:
{context}

Question: {question}

Answer:"""

retrieved = retrieve(question, top_k=2)
prompt = build_rag_prompt(question, retrieved)
print(prompt)


### Reading the Prompt

Look carefully at the printed prompt. The retrieved chunks appear before the question, and each chunk includes a citation ID such as `[embeddings:0]`.

This structure gives the generator a controlled workspace. Instead of asking the LLM to answer from memory, we are giving it a small packet of evidence and asking it to answer from that packet.

If the retrieved context is wrong, the generated answer will probably be wrong or unsupported. RAG quality starts before generation.


## 7. A Tiny Rule-Based Answerer

To keep the notebook self-contained, the next function produces a simple extractive answer from the retrieved chunks. In a real RAG system, this role is played by an LLM.

This simplified answerer is intentionally limited. It does not synthesize beautifully. It mostly concatenates retrieved text and citations. That is fine for this lesson because it lets us focus on the architecture:

```text
question -> retrieve chunks -> filter weak matches -> produce grounded response
```

Notice the `min_score` parameter. This acts like a simple refusal threshold. If all retrieved chunks have very low similarity, the system should avoid pretending it knows the answer. In production, refusal behavior is more nuanced, but the principle is the same: a RAG system needs a way to say "the retrieved evidence is not enough."


In [ ]:
def simple_grounded_answer(question, top_k=2, min_score=0.12):
    retrieved = retrieve(question, top_k=top_k)
    useful = [(score, chunk) for score, chunk in retrieved if score >= min_score]
    if not useful:
        return "I do not have enough retrieved context to answer this question."

    sentences = []
    citations = []
    for score, chunk in useful:
        sentences.append(chunk["text"])
        citations.append(f"[{chunk['chunk_id']}]")

    return " ".join(sentences) + " " + " ".join(citations)

for q in [
    "What are embeddings?",
    "How does RAG help an LLM answer questions?",
    "What is the weather tomorrow?"
]:
    print("QUESTION:", q)
    print("ANSWER:", simple_grounded_answer(q))
    print()


### Interpreting the Answers

The weather question should be refused because the document collection does not contain weather information. This is important.

A RAG system should not answer every question. It should answer when it has evidence and decline when the evidence is missing or weak.

In real systems, this behavior is usually implemented with a combination of retrieval thresholds, prompt instructions, model judgment, and evaluation rules.


## 8. Evaluation Questions

A useful RAG evaluation does not only ask whether the answer sounds good. A fluent answer can still be unsupported.

Instead, evaluate the pipeline in layers:

- **Retrieval quality:** Did the retriever find the right evidence?
- **Context quality:** Is the retrieved text enough to answer the question?
- **Groundedness:** Is every claim in the answer supported by the retrieved context?
- **Citation quality:** Do citations point to chunks that actually support the answer?
- **Refusal behavior:** Does the system say when it does not have enough evidence?
- **Answer usefulness:** Is the final response clear, concise, and appropriate for the user?

The code below checks a tiny retrieval evaluation set. It asks: does the top retrieved chunk come from the expected document?

This is a small test, but it demonstrates the habit: evaluate the retriever directly before evaluating the generated answer.


In [ ]:
evaluation_set = [
    {
        "question": "What do token embeddings do?",
        "expected_doc": "embeddings"
    },
    {
        "question": "Why does a language model need order information?",
        "expected_doc": "positional-embeddings"
    },
    {
        "question": "What components can an agentic system combine?",
        "expected_doc": "agents"
    },
]

for item in evaluation_set:
    top_score, top_chunk = retrieve(item["question"], top_k=1)[0]
    ok = top_chunk["doc_id"] == item["expected_doc"]
    print(item["question"])
    print(" expected:", item["expected_doc"])
    print(" retrieved:", top_chunk["doc_id"], f"score={top_score:.3f}", "OK" if ok else "CHECK")
    print()


### Reading the Evaluation Output

For each evaluation question, the code prints the expected document and the document retrieved as the top match.

An `OK` result means the retriever found the expected source document. A `CHECK` result does not automatically mean the system is useless; it means we should inspect the result. Sometimes the expected label is too strict. Sometimes the retriever found a related chunk that is not sufficient. Sometimes the vocabulary or chunking strategy needs improvement.

In production, evaluation sets are usually much larger and include both retrieval metrics and answer-quality metrics.


## 9. Architecture Pattern: Prompt-Only

Use prompt-only architecture when the task can be solved from the prompt itself.

```text
User -> LLM -> Response
```

This is the simplest useful LLM system. It has low engineering cost and is often enough for short, self-contained tasks.

Good fits:

- Rewrite this paragraph.
- Classify this short message.
- Explain this concept at a beginner level.
- Generate examples from information already in the prompt.

Weak fits:

- Answer from private course documents.
- Cite exact sources.
- Use recent information.
- Perform reliable calculations or external actions.

The design principle is: start here if the task allows it. Do not add retrieval, tools, or agents just because they sound advanced.


## 10. Architecture Pattern: RAG

Use RAG when the answer should depend on external documents.

```text
Documents -> chunks -> embeddings -> vector index
User question -> question embedding -> retrieve chunks -> LLM -> grounded answer
```

RAG is often the first serious architecture upgrade from prompt-only systems. It is especially useful for course assistants, policy assistants, technical documentation assistants, and enterprise knowledge systems.

A RAG system has two phases:

| Phase | What happens |
| --- | --- |
| Indexing | Documents are cleaned, chunked, embedded, and stored. |
| Query time | The question is embedded, relevant chunks are retrieved, and the LLM answers with context. |

Common RAG failure modes:

- Bad chunking hides the answer.
- The retriever returns related but unsupported context.
- The prompt does not force the model to stay grounded.
- The document collection does not contain the answer.
- The answer cites chunks that do not actually support the claim.


## 11. Architecture Pattern: Tool-Using Agent

Use tools when the system needs actions or deterministic computation.

A tool is an external capability the model can call. Examples:

- Search a database.
- Call an API.
- Run code.
- Calculate exact values.
- Create or modify files.
- Query a calendar or business system.

```text
User -> LLM decides tool call -> Tool result -> LLM response
```

Tools are different from RAG. RAG mostly retrieves textual evidence. Tools can perform actions or return structured results. A calculator, SQL query, spreadsheet operation, or file-writing function is a tool.

Tool use is appropriate when language alone is not reliable enough. For example, an LLM can explain arithmetic, but a calculator should perform arithmetic.


## 12. Architecture Pattern: Planner-Executor

Use planner-executor when the task naturally decomposes into steps.

```text
User goal -> Planner -> Step list -> Executor -> Progress and result
```

The planner decides what needs to be done. The executor performs the steps. Sometimes the same model does both roles; sometimes they are separate components.

This pattern is useful for:

- Multi-step research.
- Building or modifying software.
- Preparing reports.
- Solving tasks where intermediate progress should be inspected.

The benefit is control. The system can show the plan, track progress, recover from partial failures, and make the workflow easier to review.

The risk is overhead. For simple tasks, planning can be slower and more complicated than a direct answer.


## 13. Architecture Pattern: Reflection or Evaluator Loop

Use an evaluator loop when quality matters and the first answer is not enough.

```text
Draft -> Evaluate -> Revise -> Final
```

The evaluator can be:

- Another model call.
- A rules-based checker.
- A test suite.
- A retrieval-grounded verifier.
- A human review step.

This pattern is useful when the system needs to catch errors before returning the final result. For example, a RAG assistant can generate an answer, then check whether every sentence is supported by retrieved citations.

The key idea is that generation and evaluation are separate responsibilities. A model that writes a fluent answer is not automatically the best judge of whether the answer is grounded.


## 14. Architecture Pattern: Multi-Agent Workflow

Use multi-agent workflows when different roles can work independently or in parallel.

Examples:

- Researcher and writer.
- Planner and implementer.
- Generator and critic.
- Retriever and synthesizer.
- Specialist agents for different tools or domains.

Multi-agent designs can be powerful, but they add coordination cost. They require decisions about memory, communication, stopping conditions, conflict resolution, and final authority.

A good rule of thumb:

> Do not use multiple agents just because the task is interesting. Use multiple agents when role separation makes the system more reliable, more scalable, or easier to inspect.


## 15. Choosing the Pattern

A practical decision ladder:

| Need | Pattern |
| --- | --- |
| Simple task, no external data | Prompt-only |
| Needs documents | RAG |
| Needs actions or exact computation | Tool use |
| Needs multi-step execution | Planner-executor |
| Needs quality control | Reflection/evaluator loop |
| Needs role separation or parallel work | Multi-agent workflow |

Architecture should follow the task. More components do not automatically mean a better system.

When designing an LLM application, ask:

1. What does the user need the system to do?
2. What information does the system need?
3. Does that information fit in the prompt?
4. Does the system need external actions or tools?
5. Does the task need planning, evaluation, or multiple roles?
6. How will we know when the system failed?


## 16. Mini Lab 2: Build Your Own Tiny RAG

Choose a small document set:

- Five course paragraphs.
- A company FAQ.
- A policy document.
- Notes from another class.
- A short technical manual.

Then:

1. Create chunks.
2. Build local embeddings.
3. Retrieve top chunks for five questions.
4. Compose prompts with citations.
5. Identify one success and one failure.
6. Propose one retrieval improvement.

Deliverable suggestion:

| Question | Top retrieved chunk | Was it useful? | Why? |
| --- | --- | --- | --- |
| Example question | `[doc:0]` | Yes/No | Short explanation |

The most important part is the failure analysis. A good RAG engineer learns by inspecting mismatches between the user's question, the retrieved evidence, and the final answer.


## 17. Production Extensions

Once students understand the transparent version, the production version swaps components:

| Teaching component | Production component |
| --- | --- |
| Bag-of-words vectors | Embedding model |
| Python list of chunks | Vector database |
| Manual prompt string | Prompt template and orchestration layer |
| Rule-based answerer | LLM call |
| Manual checks | Evaluation suite and traces |

The architecture is the same. The components become stronger.

Production questions to consider:

- Which documents should be indexed?
- How often should the index be updated?
- What chunking strategy preserves meaning and citations?
- Which embedding model works best for the language and domain?
- How many chunks should be retrieved?
- Should retrieval use metadata filters?
- How should the system respond when evidence is weak?
- How will hallucinations and unsupported citations be detected?

RAG is not a single algorithm. It is an architecture pattern with many design choices.


## 18. Summary

Key takeaways:

- RAG connects retrieval with generation.
- A RAG system is usually built from documents, chunks, embeddings, a retriever, prompt composition, generation, and evaluation.
- Chunking and metadata strongly affect answer quality.
- Embeddings can be used as retrieval infrastructure, not only as internal LLM inputs.
- Retrieval quality should be inspected directly.
- Architecture patterns should be chosen by task requirements.
- More agentic complexity is useful only when it buys reliability, capability, or maintainability.

Final mental model:

```text
Prompt-only answers from the model's existing context.
RAG answers with retrieved evidence.
Tools let the system act.
Planning organizes multi-step work.
Evaluation improves quality.
Multi-agent workflows separate roles when that separation is worth the cost.
```
